In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train=pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test=pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
gender_submission=pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [3]:
PassengerId = test["PassengerId"]

train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

In [4]:
train["Age"] = train.groupby(["Sex","Pclass"])["Age"].transform(lambda x: x.fillna(x.median()))
test["Age"] = test.groupby(["Sex","Pclass"])["Age"].transform(lambda x: x.fillna(x.median()))

In [5]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1


In [6]:
train["IsAlone"] = 0
train.loc[train["FamilySize"] == 1, "IsAlone"] = 1

In [7]:
test["IsAlone"] = 0
test.loc[test["FamilySize"] == 1, "IsAlone"] = 1

In [8]:

train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

In [9]:
rare_titles = ['Don','Rev','Dr','Mme','Ms','Major','Lady','Sir','Mlle','Col','Capt','Countess','Jonkheer']

train["Title"] = train["Title"].replace(rare_titles,"Rare")
test["Title"] = test["Title"].replace(rare_titles,"Rare")

In [10]:
train["Deck"] = train["Cabin"].str[0]
test["Deck"] = test["Cabin"].str[0]


In [11]:
train["Deck"] = train["Deck"].fillna("U")
test["Deck"] = test["Deck"].fillna("U")

In [12]:
train["Sex"] = train["Sex"].map({"female":1,"male":0})
test["Sex"] = test["Sex"].map({"female":1,"male":0})


In [13]:
train = pd.get_dummies(train, columns=["Embarked","Title","Deck"], drop_first=True)
test = pd.get_dummies(test, columns=["Embarked","Title","Deck"], drop_first=True)

In [14]:
train = train.drop(["PassengerId","Name","Ticket","Cabin","SibSp","Parch"], axis=1)
test = test.drop(["PassengerId","Name","Ticket","Cabin","SibSp","Parch"], axis=1)

In [15]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

y = train["Survived"]
X = train.drop("Survived", axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

rf.fit(X_train,y_train)

from sklearn.metrics import accuracy_score

y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.8379888268156425

In [17]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

y = train["Survived"]
X = train.drop("Survived", axis=1)

In [18]:
test = test.drop("Survived", axis=1, errors="ignore")

In [19]:
rf.fit(X, y)
predictions = rf.predict(test)

In [20]:
submission = pd.DataFrame({
    "PassengerId": PassengerId,
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)